# Capítulo 3 — Flujo de líquidos por tuberías

> **Fuente.** Métodos y datos de *Transporte de Hidrocarburos por Ductos*, F. Garaicochea P.,
> C. Bernal H. y O. López O. (CIPM, 1991; reed. 2022), verificados contra la tesis UNAM de 1990 que
> le dio origen (Bernal y López, asesor Garaicochea). Los enunciados están **parafraseados**; consulte la
> obra original para el texto completo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pint
from ductos import friccion as fr
from ductos.liquidos import (Linea, PozoInyector, DatosGraper, diametro, gasto,
                             diametro_optimo_graper, costo_total_milla, dp_friccion,
                             factor_friccion, potencia_por_milla, dp_arranque_gel)

## 3.1 Ecuación general en unidades prácticas

Despreciando la aceleración, la caída de presión total es elevación + fricción:

$$\Delta p_T = 0.433\,\gamma_L\,\Delta h + 0.06056\,\frac{f\,\gamma_L\,q^2 L}{E^2 d^5}\qquad(3.23)$$

con $\Delta p$ [lb/pg²], $\Delta h$ [pies], $q$ [bl/día], $L$ [millas], $d$ [pg] y $E$ la eficiencia de flujo (3.3).

### Comprobación de las constantes con análisis dimensional

Se evalúa Darcy-Weisbach con $f=\gamma=q=L=d=1$ en unidades prácticas; el resultado es el coeficiente.

In [ ]:
u = pint.UnitRegistry()
rho = 62.428 * u.lb / u.ft**3
q = 1 * u.oil_barrel / u.day
d = 1 * u.inch
A = np.pi * d**2 / 4
v = (q / A).to(u.ft / u.s)
for milla in (5277, 5280):
    L = milla * u.ft
    dp = (rho * v**2 * L / (2 * d)).to(u.psi)
    print(f"1 milla = {milla} pie  ->  coeficiente = {dp.magnitude:.5f}")

mu = 1 * u.cP
nre = (rho * v * d / mu).to_base_units()
print(f"Coeficiente del número de Reynolds (Ec. 3.22): {nre.magnitude:.2f}")

El libro obtiene 0.06056 porque convierte la milla con 5277 pies. El valor exacto con 5280 es 0.06060
(0.06 % mayor). El paquete usa el del libro para reproducir los ejemplos.

## 3.4.1 Ejemplo: caída de presión por fricción

**Planteamiento (parafraseado).** Una línea de 3000 pies, 3.937 pg de diámetro interno y rugosidad 0.0006 pg
transporta un aceite de densidad relativa 0.9 y viscosidad 46 cp. Calcular la caída por fricción
para (a) 2560 bl/día y (b) 12 800 bl/día.

In [ ]:
linea = Linea(d=3.937, L=3000 / 5277, rug=0.0006)
for q, ref in [(2560, 11.7), (12800, 192.8)]:
    r = linea.dp(q, gamma=0.9, mu=46)
    print(f"q = {q:6d} bl/d  NRe = {r['NRe']:7.0f} ({r['regimen']:10s})  f = {r['f']:.5f}  "
          f"Δpf = {r['dp_f']:6.1f} lb/pg²   [libro: {ref}]")

**Iteraciones de Colebrook del inciso (b)**, partiendo de $f_s=0.03$ como en el libro:

In [ ]:
pd.DataFrame({"iteración": range(1, 6),
              "f calculado": fr.f_colebrook_iteraciones(5865, 0.0006 / 3.937)}).round(5)

## 3.4.2 Diseño de tuberías

Las Ecs. 3.25 y 3.26 despejan el diámetro o el gasto; como $f$ depende de ambos, se resuelven por
sustituciones sucesivas.

**Ejercicio propuesto.** ¿Qué diámetro mínimo transporta 12 800 bl/día del aceite anterior a lo largo de
5 millas con una caída disponible de 500 lb/pg² y una subida de 150 pies? ¿Qué gasto pasaría por una
línea comercial de 6 pg (d.i. 6.065 pg) con esa misma caída?

In [ ]:
d_min = diametro(q=12800, gamma=0.9, mu=46, L=5, dp_T=500, dh=150)
q_6 = gasto(d=6.065, gamma=0.9, mu=46, L=5, dp_T=500, dh=150)
print(f"Diámetro mínimo: {d_min:.3f} pg")
print(f"Gasto en 6 pg cédula 40: {q_6:,.0f} bl/día")
# Comprobación: el gasto obtenido reproduce la caída disponible
print(Linea(d=6.065, L=5, dh=150).dp(q_6, 0.9, 46)["dp_T"])

## 3.4.3–3.4.4 Pozo inyector de agua: análisis nodal

El sistema tiene cuatro elementos (estación de bombeo → línea superficial → tubería vertical →
yacimiento). Para un gasto supuesto se recorren los nodos:

1. $p_{th} = p_p - \Delta p_{línea}$ (Ec. 3.23 con el desnivel de la línea)
2. $p_{wf} = p_{th} + 0.433\gamma\,\Delta h - \Delta p_f$ (la columna ayuda, la fricción se opone)
3. $p_{ws} = p_{wf} - q/II$ (Ec. 3.27)
4. Se ajusta $q$ hasta que $p_{ws}$ calculada coincida con la real.

**Datos del ejemplo.** Línea: agua, 2 millas, 4 pg, 1 cp, E = 0.90, sube 20 pies.
Tubería vertical: 2.441 pg, 12 000 pies (2.273 millas), 0.8 cp, E = 0.90.
$p_p$ = 3100 lb/pg², $II$ = 3.521 bl/día/(lb/pg²), $p_{ws}$ = 4402 lb/pg², ε = 0.0006 pg.

> El diámetro de la línea no es legible en la tesis escaneada; 4 pg es el que reproduce la Tabla 3.1
> y coincide con el caso base de la Tabla 3.2.

In [ ]:
def pozo(d_linea=4.0, d_tp=2.441):
    return PozoInyector(
        linea=Linea(d=d_linea, L=2.0, dh=20.0, E=0.9),
        tp=Linea(d=d_tp, L=2.273, dh=-12000.0, E=0.9),
        gamma_linea=1.0, mu_linea=1.0, gamma_tp=1.0, mu_tp=0.8, ii=3.521)

libro = {6000: (2997.8, 7011.3, 5307.2), 8000: (2932.5, 6084.4, 3812.3),
         10000: (2851.0, 4913.1, 2073.0), 7250: (2959.0, 6460.7, 4401.6)}
filas = []
for q, (pth, pwf, pws) in libro.items():
    r = pozo().nodos(q, 3100)
    filas.append({"q supuesto": q, "pth": r["pth"], "pth libro": pth,
                  "pwf": r["pwf"], "pwf libro": pwf, "pws": r["pws"], "pws libro": pws})
pd.DataFrame(filas).round(1)

In [ ]:
q_iny = pozo().gasto_inyeccion(3100, 4402)
qs = np.linspace(4000, 11000, 40)
pws = [pozo().nodos(q, 3100)["pws"] for q in qs]
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(qs, pws, label="pws calculada")
ax.axhline(4402, ls="--", c="k", lw=1, label="pws real")
ax.plot(q_iny, 4402, "o", c="C3")
ax.annotate(f"q = {q_iny:,.0f} bl/d", (q_iny, 4402), xytext=(10, 15), textcoords="offset points")
ax.set(xlabel="Gasto de inyección (bl/día)", ylabel="pws (lb/pg²)",
       title="Fig. 3.3 — Gasto de inyección por análisis nodal")
ax.grid(alpha=0.3); ax.legend(); plt.show()
print(f"Gasto de inyección: {q_iny:.0f} bl/día  [libro: 7250, leído de la gráfica]")

### Sensibilidad (Tabla 3.2, Figs. 3.4 a 3.6)

Efecto del diámetro de la línea de descarga, del de la tubería vertical y de la presión de bombeo.

In [ ]:
casos = {
    "Línea de descarga (pg)": ([2, 3, 4, 5], [4690, 6765, 7250, 7365], lambda x: pozo(d_linea=x).gasto_inyeccion(3100, 4402)),
    "Tubería vertical (pg)": ([2, 3, 4, 5], [5105, 9422, 11400, 12050], lambda x: pozo(d_tp=x).gasto_inyeccion(3100, 4402)),
    "Presión de bombeo (lb/pg²)": ([2000, 3100, 4000, 5000], [5710, 7250, 8375, 9530], lambda x: pozo().gasto_inyeccion(x, 4402)),
}
fig, axs = plt.subplots(1, 3, figsize=(14, 4))
for ax, (nombre, (xs, ref, fn)) in zip(axs, casos.items()):
    calc = [fn(x) for x in xs]
    ax.plot(xs, calc, "o-", label="calculado")
    ax.plot(xs, ref, "s--", mfc="none", label="libro")
    ax.set(xlabel=nombre, ylabel="q (bl/día)"); ax.grid(alpha=0.3); ax.legend()
    print(nombre, [f"{c:,.0f}" for c in calc])
plt.tight_layout(); plt.show()

La selección de diámetros debe justificarse comparando el gasto adicional con la inversión adicional,
que es justamente el tema de la sección 3.5.

## 3.4.5 Pérdidas en conexiones

Cada accesorio se sustituye por una longitud equivalente de tubería recta que se suma a la longitud real
(nomograma de la Fig. 3.7; p. ej., un codo de 90° de 2 pg equivale a ~12 pies). *Pendiente:* tabla
digital de $L_e/d$ para accesorios comunes.

## 3.5 Optimización de líneas de conducción de líquidos (método de Graper)

El costo total a valor presente por milla combina inversión y operación de la tubería y de la estación de bombeo:

$$C_t(d) = Z_v\left[(I_{p1}d + I_{p2})\,d + X_s\,\mathrm{BHP}\right] + W\left[Y_p\,d + Y_x\,\mathrm{BHP}\right],\qquad \mathrm{BHP} = \frac{B}{d^5}$$

con $Y_x = Y_f + Y_s F_c$. Derivando e igualando a cero:

$$d^6 = \frac{5B\,(Z_v X_s + W Y_x)}{Z_v(2I_{p1}d + I_{p2}) + W Y_p}$$

que se resuelve por sustituciones sucesivas actualizando $f$ en cada paso (Fig. 3.9).

### 3.5.2 Ejemplo: red de distribución de gasolina

Refinería → línea troncal → nodo de unión → terminales A (15 000 bl/d), B (40 000) y C (100 000).
Gasolina: 0.525 cp, densidad relativa 0.7. Datos económicos de la Tabla 3.5.

In [ ]:
eco = DatosGraper()
print(eco)
print(f"Yx = Yf + Ys·Fc = {eco.yx} dll/BHP/año")
libro_36 = {15000: (5.536, 28.048, 8.957), 40000: (8.357, 22.978, 19.568),
            100000: (12.267, 19.273, 41.034), 155000: (14.734, 17.764, 58.621)}
filas = []
for q, (d, dp, bhp) in libro_36.items():
    r = diametro_optimo_graper(q, gamma=0.7, mu=0.525, eco=eco)
    filas.append({"q (bl/d)": q, "d óptimo": r["d"], "libro": d, "Δp/milla": r["dp_milla"],
                  "libro Δp": dp, "BHP/milla": r["bhp_milla"], "libro BHP": bhp,
                  "iteraciones": r["iteraciones"]})
pd.DataFrame(filas).round(3)

Tabla 3.6 reproducida. La curva de costo total contra diámetro (Fig. 3.14) muestra el mínimo:

In [ ]:
ds = np.linspace(3, 24, 400)
fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
for q in libro_36:
    ct = np.array([costo_total_milla(d, q, 0.7, 0.525, eco) for d in ds]) / 1e3
    dopt = diametro_optimo_graper(q, 0.7, 0.525, eco)["d"]
    cmin = costo_total_milla(dopt, q, 0.7, 0.525, eco) / 1e3
    line, = axs[0].plot(ds, ct, label=f"{q/1000:.0f} Mbl/d")
    axs[0].plot(dopt, cmin, "o", c=line.get_color())
    axs[1].plot(ds, ct / cmin, c=line.get_color())
    axs[1].plot(dopt, 1, "o", c=line.get_color())
axs[0].set(xlabel="Diámetro interno (pg)", ylabel="Costo total VP (miles dll/milla)",
           ylim=(0, 450), title="Fig. 3.14 — Costo total vs. diámetro")
axs[1].set(xlabel="Diámetro interno (pg)", ylabel="Costo / costo mínimo", ylim=(0.95, 1.6),
           title="Curvas normalizadas: el mínimo es plano")
for ax in axs: ax.grid(alpha=0.3)
axs[0].legend(); plt.tight_layout(); plt.show()

Cerca del óptimo la curva es plana: desviarse ±0.5 pg cambia el costo solo unos puntos porcentuales, lo que da margen para elegir el diámetro comercial.

### Selección de diámetros comerciales y balance hidráulico (Tablas 3.4, 3.7 y 3.9)

Se toma el diámetro comercial inmediato superior (API 5LX X-46, Tabla 3.4) y se recalculan presiones
con 30 lb/pg² de descarga en cada terminal. La presión en el nodo de unión la fija la rama más exigente;
las demás ramas tendrían que estrangularse.

In [ ]:
comerciales = pd.DataFrame({
    "d_ext": [6.625, 8.625, 10.750, 12.750, 14, 16, 18],
    "d_int": [6.375, 8.313, 10.313, 12.250, 13.438, 15.374, 17.313],
    "costo_milla": [70000, 100000, 140000, 190000, 220000, 275000, 345000]}).set_index("d_ext")

def dp_milla(q, d_ext):
    di = comerciales.loc[d_ext, "d_int"]
    return dp_friccion(q, 0.7, di, 1.0, factor_friccion(q, 0.7, di, 0.525, 0.0006))

ramas = {"A": (15000, 25), "B": (40000, 15), "C": (100000, 20)}
troncal = (155000, 25, 16)

def evaluar(diametros):
    req = {k: dp_milla(q, diametros[k]) * L + 30 for k, (q, L) in ramas.items()}
    p_union = max(req.values())
    p_ref = p_union + dp_milla(troncal[0], troncal[2]) * troncal[1]
    bhp = potencia_por_milla(troncal[0], p_ref, eco.e)
    return req, p_union, p_ref, bhp

alternativas = {"1: diámetros óptimos": {"A": 6.625, "B": 10.75, "C": 14},
                "2: A de 6 a 8 pg": {"A": 8.625, "B": 10.75, "C": 14},
                "3: B de 10 a 8 pg": {"A": 6.625, "B": 8.625, "C": 14}}
res = {}
for nombre, dmt in alternativas.items():
    req, pu, pr, bhp = evaluar(dmt)
    res[nombre] = bhp
    print(f"{nombre:22s} req: " + ", ".join(f"{k}={v:5.0f}" for k, v in req.items())
          + f" | unión {pu:4.0f} | refinería {pr:4.0f} lb/pg² | {bhp:5.0f} BHP")
print("Libro: 2442, 2099 y 2455 BHP")

### 3.5.3 Toma de decisiones

Se comparan, a valor presente, el cambio de costo en tubería contra el cambio en la estación de bombeo.

In [ ]:
def delta_costos(rama, d_antes, d_despues, L, bhp_antes, bhp_despues):
    c = comerciales
    tub = ((c.loc[d_despues, "costo_milla"] - c.loc[d_antes, "costo_milla"]) * L * eco.zv
           + (c.loc[d_despues, "d_int"] - c.loc[d_antes, "d_int"]) * eco.yp * L * eco.w)
    dbhp = bhp_despues - bhp_antes
    bomba = dbhp * eco.xs * eco.zv + dbhp * eco.yx * eco.w
    return tub, bomba

b1, b2, b3 = res.values()
for etiqueta, args, bhp in [("Alt. 2 (A 6→8 pg)", ("A", 6.625, 8.625, 25), b2),
                            ("Alt. 3 (B 10→8 pg)", ("B", 10.75, 8.625, 15), b3)]:
    tub, bomba = delta_costos(*args, b1, bhp)
    print(f"{etiqueta}: tubería {tub:+,.0f} dll | bombeo {bomba:+,.0f} dll | neto {tub + bomba:+,.0f} dll")
print("Libro: Alt. 2 tubería +686,938 / bombeo -485,588; Alt. 3 tubería -548,626 / bombeo +18,401")

La alternativa 3 ahorra dinero y además reduce el desbalance de presiones en el nodo de unión: la línea a B
de 10 pg tendría que estrangularse, mientras que con 8 pg solo sube 4 lb/pg² la presión en la refinería.

## 3.5.5 Flujo de crudos viscosos: arranque con gel

Para romper el gel de un crudo parafínico después de un paro (Ec. 3.42), con factor de seguridad de 1.5 a 2:

In [ ]:
for tau in [10, 50, 100]:
    print(f"τ = {tau:3d} lbf/100 pie² en 20 millas de 12 pg:  "
          f"Δp = {dp_arranque_gel(tau, 20, 12):6.0f} lb/pg²   "
          f"(con FS = 2: {dp_arranque_gel(tau, 20, 12, 2.0):6.0f})")